# Narratives (ds002345) - Preprocessing

Now uses the SAME volumetric pipeline as LPP and NNDb, so all three datasets are identical:
download -> parcellate cortex (1000) -> parcellate subcortex (32) -> clean -> resample to 1 Hz
-> save -> verify -> delete raw.

(Narratives' brain file is the MNI152NLin2009cAsym volume; subjects live under derivatives/fmriprep/.)

## Setup

In [ ]:
import os, sys

SUBJECTS = [f"sub-{i:03d}" for i in range(1, 346)]  # <-- edit this list (e.g. [f"sub-{i:03d}" for i in range(1, 21)])

PROJECT      = r"C:\TRIBE_Preprocessing"
DATASET      = "narratives"
DATASET_ROOT = os.path.join(PROJECT, "data", DATASET)
ATLAS_DIR    = os.path.join(PROJECT, "atlases")
OUTPUTS_BASE = os.path.join(PROJECT, "outputs", DATASET)

sys.path.append(PROJECT)
import helpers.helpers_narratives as H
print("Subjects:", SUBJECTS)

## Load the two atlases once (cortex = 1000, subcortex = 32)

In [ ]:
cortical_atlas    = H.get_cortical_atlas()
subcortical_atlas = H.get_subcortical_atlas(ATLAS_DIR)
H.enable_fallback_remote(DATASET_ROOT)
print("Atlases ready.")

## Run the batch

In [ ]:
for SUBJECT in SUBJECTS:
    print(f"\n=== {SUBJECT} ===")
    OUT_DIR = os.path.join(OUTPUTS_BASE, SUBJECT)
    try:
        func_dir = H.download_subject(DATASET_ROOT, SUBJECT)
        runs = H.find_runs(func_dir, SUBJECT)
        print(f"  {len(runs)} run(s): {[r['name'] for r in runs]}")

        for run in runs:
            cortex    = H.parcellate(run["bold"], cortical_atlas)
            subcortex = H.parcellate(run["bold"], subcortical_atlas)
            cortex    = H.resample_1hz(H.clean(cortex,    run["tr"]), run["tr"])
            subcortex = H.resample_1hz(H.clean(subcortex, run["tr"]), run["tr"])

            tag = f"{SUBJECT}_{run['name']}"
            H.save_npy(OUT_DIR, f"{tag}_cortical_parcels.npy",    cortex)
            H.save_npy(OUT_DIR, f"{tag}_subcortical_parcels.npy", subcortex)
            H.save_metadata(OUT_DIR, f"{tag}_metadata.json", {
                "subject": SUBJECT, "dataset": DATASET, "run": run["name"],
                "native_tr_seconds": run["tr"], "resampled_hz": 1,
                "hemodynamic_lag_seconds": 5,
                "shapes": {"cortical_parcels": list(cortex.shape),
                           "subcortical_parcels": list(subcortex.shape)},
            })
            del cortex, subcortex

        if H.outputs_complete(OUT_DIR, len(runs)):
            print("  outputs verified -> cleaning up")
            H.cleanup_subject(DATASET_ROOT, SUBJECT)
        else:
            print(f"  !! outputs incomplete for {SUBJECT} - keeping raw data")
    except Exception as e:
        print(f"  !! {SUBJECT} FAILED: {e}\n     keeping raw data; moving on.")

print("\n=== BATCH COMPLETE ===")

## Summary

In [ ]:
for SUBJECT in SUBJECTS:
    d = os.path.join(OUTPUTS_BASE, SUBJECT)
    n = len([f for f in os.listdir(d) if f.endswith('.npy')]) if os.path.isdir(d) else 0
    print(f"{SUBJECT}: {n} .npy files")